## Modern CNN

### 1. AlexNet (2012)

<img src="img/lnetalex.png" width=500 height=500 />

#### Capacity Control and Preprocessing

- AlexNet controls the model complexity of the fully connected layer by dropout, while LeNet only uses weight decay.

- To augment the data even further, the training loop of AlexNet added a great deal of image augmentation, such as flipping, clipping, and color changes. This makes the model more robust and the larger sample size effectively reduces overfitting

In [40]:
import torch
import torch.nn as nn
from torchvision import models
from torchsummary import summary
import torchinfo

In [41]:
alex_model = models.alexnet(pretrained=True)
torchinfo.summary(alex_model,(3, 224, 224),batch_dim = 0)

Layer (type:depth-idx)                   Output Shape              Param #
AlexNet                                  [1, 1000]                 --
├─Sequential: 1-1                        [1, 256, 6, 6]            --
│    └─Conv2d: 2-1                       [1, 64, 55, 55]           23,296
│    └─ReLU: 2-2                         [1, 64, 55, 55]           --
│    └─MaxPool2d: 2-3                    [1, 64, 27, 27]           --
│    └─Conv2d: 2-4                       [1, 192, 27, 27]          307,392
│    └─ReLU: 2-5                         [1, 192, 27, 27]          --
│    └─MaxPool2d: 2-6                    [1, 192, 13, 13]          --
│    └─Conv2d: 2-7                       [1, 384, 13, 13]          663,936
│    └─ReLU: 2-8                         [1, 384, 13, 13]          --
│    └─Conv2d: 2-9                       [1, 256, 13, 13]          884,992
│    └─ReLU: 2-10                        [1, 256, 13, 13]          --
│    └─Conv2d: 2-11                      [1, 256, 13, 13]         

### 2. Networks Using Blocks (VGG) (2014)

The basic building block of CNNs is a sequence of the following: 

    - (i) a convolutional layer with padding to maintain the resolution, 
    - (ii) a nonlinearity such as a ReLU, 
    - (iii) a pooling layer such as max-pooling to reduce the resolution. 

One of the problems with this approach is that **the spatial resolution decreases quite rapidly**. 

In particular, this imposes a hard limit of $log_{2}$ $d$ convolutional layers on the network before all dimensions $(d)$ are used up.

>> For instance, in the case of ImageNet, it would be impossible to have more than 8 convolutional layers in this way.

**The key idea of VGG** was to use multiple convolutions in between downsampling via max-pooling in the form of a block

**VGG block** consists of a sequence of convolutions with 3 × 3 kernels with padding of 1 (keeping height and width) followed by a 2 × 2 max-pooling layer with stride of 2 (halving height and width after each block)

### VGG Network

the VGG Network can be partitioned into two parts: 

- the first consisting mostly of convolutional and pooling layers

- and the second consisting of fully connected layers that are identical to those in AlexNet


**The key difference** is that the convolutional layers are grouped in nonlinear transformations that leave the dimensonality unchanged, followed by a resolution-reduction step

<img src="img/vgg.png" width=500 height=500 />

In [36]:
vgg16_model = models.vgg16(pretrained=True)

torchinfo.summary(vgg16_model,(3, 224, 224),batch_dim = 0)

Layer (type:depth-idx)                   Output Shape              Param #
VGG                                      [1, 1000]                 --
├─Sequential: 1-1                        [1, 512, 7, 7]            --
│    └─Conv2d: 2-1                       [1, 64, 224, 224]         1,792
│    └─ReLU: 2-2                         [1, 64, 224, 224]         --
│    └─Conv2d: 2-3                       [1, 64, 224, 224]         36,928
│    └─ReLU: 2-4                         [1, 64, 224, 224]         --
│    └─MaxPool2d: 2-5                    [1, 64, 112, 112]         --
│    └─Conv2d: 2-6                       [1, 128, 112, 112]        73,856
│    └─ReLU: 2-7                         [1, 128, 112, 112]        --
│    └─Conv2d: 2-8                       [1, 128, 112, 112]        147,584
│    └─ReLU: 2-9                         [1, 128, 112, 112]        --
│    └─MaxPool2d: 2-10                   [1, 128, 56, 56]          --
│    └─Conv2d: 2-11                      [1, 256, 56, 56]          29

### 3. Network in Network (NiN) (2013)

LeNet, AlexNet, and VGG all share a common design pattern: 

**extract features exploiting spatial structure via a sequence of convolutions and pooling layers and post-process the representations via fully connected layers**

##### This design poses two major challenges. 

- First, the fully connected layers at the end of the architecture consume tremendous numbers of parameters (not suitable for mobile and embedded devices)

- Second, it is equally impossible to add fully connected layers earlier in the network to increase the degree of nonlinearity: doing so would destroy the spatial structure and require potentially even more memory.

**The network in network (NiN) blocks** offer an alternative, capable of solving both problems in one simple strategy.

They were proposed based on a very simple insight: 

- (i) use 1 × 1 convolutions to add local nonlinearities across the channel activations and
  
- (ii) use global average pooling to integrate across all locations in the last representation layer.


The idea behind NiN is to apply a fully connected layer at each pixel location (for each height and width). The resulting 1×1 convolution can be thought of as a fully connected layer acting independently on each pixel location.

<img src="img/nin.png" width=500 height=500 />

In [42]:
def nin_block(out_channels, kernel_size, strides, padding):
    return nn.Sequential(
        nn.LazyConv2d(out_channels, kernel_size, strides, padding),
        nn.ReLU(),
        nn.LazyConv2d(out_channels, kernel_size=1), 
        nn.ReLU(),
        nn.LazyConv2d(out_channels, kernel_size=1), 
        nn.ReLU())

In [48]:
class NiN(nn.Module):
    def __init__(self, lr=0.1, num_classes=10):
        super().__init__()
        
        self.net = nn.Sequential(
            nin_block(96, kernel_size=11, strides=4, padding=0),
            nn.MaxPool2d(3, stride=2),
            nin_block(256, kernel_size=5, strides=1, padding=2),
            nn.MaxPool2d(3, stride=2),
            nin_block(384, kernel_size=3, strides=1, padding=1),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout(0.5),
            nin_block(num_classes, kernel_size=3, strides=1, padding=1),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten())

### 4. Multi-Branch Networks (GoogLeNet) (2015)

- It uses a structure that combined the strengths of NiN, repeated blocks, and a cocktail of convolution kernels. 

- It was arguably the first network that exhibited a clear distinction among the stem (data ingest), body (data processing), and head (prediction) in a CNN

- **The key contribution in GoogLeNet** was the design of the network body.

    - It solved the problem of selecting convolution kernels (1 × 1 to 11 × 11 ) in an ingenious way. It simply concatenated multi-branch convolutions.

### Inception Blocks

<img src="img/inception.png" width=500 height=500 />

- The first three branches use convolutional layers with window sizes of 1 × 1, 3 × 3, and 5 × 5 **to extract information from different spatial sizes.**
-  The middle two branches also add a 1 × 1 convolution of the input to reduce the number of channels, **reducing the model’s complexity.**
-  The fourth branch uses a 3 × 3 max-pooling layer, followed by a 1 × 1 convolutional layer to change the number of channels.

- The four branches all use appropriate padding to give the input and output the same height and width.
- The outputs along each branch are concatenated along the channel dimension and comprise the block’s output.
- The commonly-tuned hyperparameters of the Inception block are the number of output channels per layer, i.e., how to allocate capacity among convolutions of different size.

**To gain some intuition for why this network works so well, consider the combination of the filters.**

#### GoogLeNet Model

<img src="img/googlelenet.png" width=500 height=500 />

## 5. Residual Networks (ResNet) and ResNeXt

When design deeper networks, what important is the ability to design networks where adding layers makes networks strictly **more expressive rather than just different.**

#### Function Classes

given a dataset with features $X$ and labels $y$, we might try finding $ f_{F}^{*} $ by solving the following optimization problem:

<img src="img/funclass.png" width=300 height=300 />

<img src="img/nest.png" width=500 height=500 />


- If larger function classes contain the smaller ones we are guaranteed that increasing them strictly increases the expressive power of the network.

- **For deep neural networks**, if we can train the newly-added layer into an identity function $f(x) = x$, the new model will be as effective as the original model. As the new model may get a better solution to fit the training dataset, the added layer might make it easier to reduce training errors.

  
- **The idea behind residual network** (ResNet) is that every additional layer should more easily contain **the identity function** as one of its elements.

### Residual Blocks

<img src="img/res.png" width=500 height=500 />

- The right figure illustrates the **residual block of ResNet**, where the solid line carrying the layer input x to the addition operator is called a **residual connection** (or shortcut connection).

- The portion within the dotted-line box needs to learn **the residual mapping** $g (x) = f (x)-x$ making **the identity mapping** $f (x) = x$ easier to learn.

- the residual block can be thought of as a special case of the multi-branch Inception block: it has two branches one of which is the identity mapping.

#### ResNet block with and without 1 × 1 convolution

<img src="img/resnet.png" width=500 height=500 />

#### ResNet Model

<img src="img/resnetm.png" width=500 height=500 />

### ResNeXt

- One of the challenges one encounters in the design of ResNet is the trade-off between non-linearity and dimensionality within a given block.
    - meaning, we could add more nonlinearity by increasing the number of layers, or by increasing the width of the convolutions
    - An alternative strategy is to increase the number of channels that can carry information between blocks. But this technique comes with a quadratic penalty.

<br>

- Inspiration from the Inception block, **ResNeXt applies multiple independent groups to the ResNet block**
- Different from the smorgasbord of transformations in Inception, **ResNeXt** adopts the same transformation in all branches, thus minimizing the need for manual tuning of each branch.


<img src="img/resnext.png" width=500 height=500 />


- Breaking up a convolution from $c_{i}$ to $c_{o}$ channels into one of $g$ groups of size $c_{i}/g$ generating $g$ outputs of size $c_{o}/g$ is called, **a grouped convolution**
- The computational cost is reduced from $O(c_{i}. c_{o})$ to $O(g.(c_{i}/g).(c_{o}/g))$ $=$ $O(c_{i}. c_{o} /g )$, i.e., it is $g$ times faster.
- the number of parameters needed to generate the output is also reduced from a $c_{i}$ x $c_{o}$ matrix to $g$ smaller matrices of size $(c_{i}/g)$ x $(c_{o}/g)$, again a $g$ times reduction.


- The only challenge in this design is that no information is exchanged between the $g$ groups.

**The ResNeXt block** amends this in two ways:

- the grouped convolution with a 3 × 3 kernel is sandwiched in between two 1 × 1 convolutions.

## 6. DenseNet

- ResNet decomposes functions into:


>> $f(x) = g(x) + x$


ResNet decomposes $f$ into a simple linear term and a more complex nonlinear one. 

What if we wanted to capture (not necessarily add) information beyond two terms? One such solution is DenseNet 


<img src="img/res_vs_dens.png" width=300 height=300 />


- we perform a mapping from $x$ to its values after applying an increasingly complex sequence of functions:

<img src="img/denseq.png" width=400 height=400 />


- The name DenseNet arises from the fact that the dependency graph between variables becomes quite dense. The final layer of such a chain is densely connected to all previous layers.


<img src="img/densgraph.png" width=300 height=300 />

>>> Note how the dimensionality increases with depth.




<space>
<space>
<space>

#### The main components that comprise a DenseNet are **dense blocks** and **transition layers**.

**Dense blocks**
    
    - define how the inputs and outputs are concatenated.

In [ ]:
def conv_block(num_channels):
    return nn.Sequential(
        nn.LazyBatchNorm2d(), nn.ReLU(),
        nn.LazyConv2d(num_channels, kernel_size=3, padding=1))

In [ ]:
class DenseBlock(nn.Module):
    def __init__(self, num_convs, num_channels):
        super(DenseBlock, self).__init__()
        layer = []
        for i in range(num_convs):
            layer.append(conv_block(num_channels))
        self.net = nn.Sequential(*layer)

    def forward(self, X):
        for blk in self.net:
            Y = blk(X)
            # Concatenate input and output of each block along the channels
            X = torch.cat((X, Y), dim=1)
        return X


**Transition layers**

- control the number of channels so that it is not too large, since the expansion $ x 
 \rightarrow [x, f_{1}(x), f_{2} ([x, f_{1} (x)]),...]$ can be quite high-dimensional.

- it reduces the number of channels by using a 1 × 1 convolution.
- Moreover, it halves the height and width via average pooling with a stride of 2.

In [ ]:
def transition_block(num_channels):
    return nn.Sequential(
        nn.LazyBatchNorm2d(), nn.ReLU(),
        nn.LazyConv2d(num_channels, kernel_size=1),
        nn.AvgPool2d(kernel_size=2, stride=2))

In [ ]:
class DenseNet(nn.module):
    def __init__(self, num_channels=64, growth_rate=32, arch=(4, 4, 4, 4),
             lr=0.1, num_classes=10):
        super(DenseNet, self).__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(self.b1())
        for i, num_convs in enumerate(arch):
            
            self.net.add_module(f'dense_blk{i+1}', DenseBlock(num_convs,
                                                          growth_rate))
            # The number of output channels in the previous dense block
            num_channels += num_convs * growth_rate
            
            # A transition layer that halves the number of channels is added
            # between the dense blocks
            if i != len(arch) - 1:
                num_channels //= 2
                self.net.add_module(f'tran_blk{i+1}', transition_block(
                    num_channels))
        
        self.net.add_module('last', nn.Sequential(
            nn.LazyBatchNorm2d(), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.LazyLinear(num_classes)))
        self.net.apply(nn.init.xavier_uniform_)
        
        def b1(self):
            return nn.Sequential(
                nn.LazyConv2d(64, kernel_size=7, stride=2, padding=3),
                nn.LazyBatchNorm2d(), nn.ReLU(),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1))